In [1]:
"""OSCAR download and NetCDF utilities using PO.DAAC downloader workflow."""

from __future__ import annotations

import os
import re
import stat
import subprocess
import warnings

import rasterio
from rasterio.transform import from_bounds

from dataclasses import dataclass
from datetime import date, datetime
from calendar import monthrange
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))


In [2]:
from podaac import run_podaac_downloader
from gridding import standardize_oscar_uv_netcdf, export_oscar_uv_geotiff
from periods import seasonal_periods

In [3]:
@dataclass(frozen=True)
class StudyArea:
    """Bounding box used to subset OSCAR fields."""

    lon_min: float
    lon_max: float
    lat_min: float
    lat_max: float


@dataclass(frozen=True)
class OscarDownloadConfig:
    """PO.DAAC downloader options for OSCAR data pulls."""

    output_dir: Path
    podaac_collection: str
    u_var: str = "u"
    v_var: str = "v"


def validate_study_area(area: StudyArea) -> StudyArea:
    """Validate and normalize StudyArea bounds."""
    lon_min, lon_max = sorted([float(area.lon_min), float(area.lon_max)])
    lat_min, lat_max = sorted([float(area.lat_min), float(area.lat_max)])

    if not (-180.0 <= lon_min <= 360.0 and -180.0 <= lon_max <= 360.0):
        raise ValueError(f"Invalid longitude bounds: {lon_min}, {lon_max}")
    if not (-90.0 <= lat_min <= 90.0 and -90.0 <= lat_max <= 90.0):
        raise ValueError(f"Invalid latitude bounds: {lat_min}, {lat_max}")
    if lon_min == lon_max or lat_min == lat_max:
        raise ValueError("StudyArea bounds collapse to zero width/height")

    return StudyArea(lon_min=lon_min, lon_max=lon_max, lat_min=lat_min, lat_max=lat_max)

    
def infer_study_area_from_hotspots(
    hotspot_csv: Path,
    lon_col: str = "lon",
    lat_col: str = "lat",
    pad_deg: float = 0.5,
) -> StudyArea:
    """Infer a current-download bounding box from hotspot coordinates."""
    df = pd.read_csv(hotspot_csv)
    area = StudyArea(
        lon_min=float(df[lon_col].min() - pad_deg),
        lon_max=float(df[lon_col].max() + pad_deg),
        lat_min=float(df[lat_col].min() - pad_deg),
        lat_max=float(df[lat_col].max() + pad_deg),
    )
    area = validate_study_area(area)
    print(f"[DEBUG] inferred StudyArea: {area}")
    return area


def _raw_oscar_files(output_dir: Path) -> set[Path]:
    """Track raw OSCAR .nc files only (exclude clipped outputs)."""
    return {
        p.resolve()
        for p in output_dir.glob("*.nc")
        if "clip" not in p.name.lower() and p.name.lower().startswith("oscar")
    }


def _extract_data_date(raw_file: Path) -> str:
    """Extract YYYYMMDD from a raw OSCAR filename."""
    m = re.search(r"(19|20)\d{6}", raw_file.name)
    if not m:
        raise ValueError(f"Could not parse data date from filename: {raw_file.name}")
    return m.group(0)

    
def download_oscar_for_periods(
    cfg: OscarDownloadConfig,
    area: StudyArea,
    periods: list[tuple[date, date, str]],
    *,
    standardize: bool = False,
) -> list[Path]:
    outputs: list[Path] = []
    area = validate_study_area(area)
    print(f"[DEBUG] download StudyArea: {area}")

    for pstart, pend, pid in periods:
        start_str = pstart.strftime('%Y-%m-%dT%H:%M:%SZ')
        end_str = pend.strftime('%Y-%m-%dT%H:%M:%SZ')

        downloader_bbox = None if standardize else area
        print(f"[DEBUG] period={pid} downloading...")
        run_podaac_downloader(
            collection=cfg.podaac_collection,
            output_dir=cfg.output_dir,
            start_date=start_str,
            end_date=end_str,
            bbox=downloader_bbox,
            dry_run=False,
        )

        raw_files = sorted(_raw_oscar_files(cfg.output_dir), key=lambda p: p.stat().st_mtime)
        if not raw_files:
            raise FileNotFoundError(f"No raw NetCDF files found in {cfg.output_dir} after period {pid} download")

        raw_out = raw_files[-1]  # latest
        if standardize:
            data_date = _extract_data_date(raw_out)
            std_out = cfg.output_dir / f"oscar_uv_clip_{data_date}.nc"
            if std_out.exists():
                print(f"[DEBUG] clip exists, skipping: {std_out.name}")
            else:
                print(f"[DEBUG] standardize raw={raw_out.name} -> clip={std_out.name}")
                standardize_oscar_uv_netcdf(raw_out, std_out, area, u_var=cfg.u_var, v_var=cfg.v_var)
            outputs.append(std_out)
        else:
            outputs.append(raw_out)

    return outputs

In [4]:
area = infer_study_area_from_hotspots(Path(r"C:\Users\Bruno\OneDrive - CIMA Foundation\Documenti\UNDP_Djibouti\UN projects - 2026_UNDP Djibouti\DATA\00_Hazard\MarinePollution\marine-traffic\gmtds_tanker_hotspots_multi.csv"), pad_deg=0.7)
    

[DEBUG] inferred StudyArea: StudyArea(lon_min=41.50803970311112, lon_max=45.69349690701655, lat_min=9.72568026223306, lat_max=14.679851304878618)


In [5]:
periods = seasonal_periods(
    start_date=datetime(2020, 1, 1, 0, 0, 0).strftime('%Y-%m-%dT%H:%M:%SZ'), #dates should be strings in YYYYMMDDhhmmssZ format
    end_date=datetime(2020, 2, 1, 23, 59, 59).strftime('%Y-%m-%dT%H:%M:%SZ'),
    season_length_months=1,
)
periods

C:\Users\Bruno\OneDrive - CIMA Foundation\Documenti\Python_Scripts\oilspills-risk\oilspill_risk\periods.py:38: UserWarning: Skipping incomplete season 2: [2020-02 to 2020-02] vs range [2020-01-01 to 2020-02-01]
  warnings.warn(


[(datetime.date(2020, 1, 1), datetime.date(2020, 1, 31), 'S1_202001_202001')]

In [6]:
cfg = OscarDownloadConfig(
    output_dir=Path(r"C:\Users\Bruno\OneDrive - CIMA Foundation\Documenti\UNDP_Djibouti\UN projects - 2026_UNDP Djibouti\DATA\00_Hazard\MarinePollution\oscar_currents"),
    podaac_collection="OSCAR_L4_OC_FINAL_V2.0",
)
cfg

OscarDownloadConfig(output_dir=WindowsPath('C:/Users/Bruno/OneDrive - CIMA Foundation/Documenti/UNDP_Djibouti/UN projects - 2026_UNDP Djibouti/DATA/00_Hazard/MarinePollution/oscar_currents'), podaac_collection='OSCAR_L4_OC_FINAL_V2.0', u_var='u', v_var='v')

In [7]:
#downloaded = download_oscar_for_periods(cfg, area, periods, standardize=True)

In [8]:
from datetime import timedelta

raw_files = sorted(_raw_oscar_files(cfg.output_dir), key=lambda p: p.stat().st_mtime)
if not raw_files:
    raise FileNotFoundError(f"No raw NetCDF files found in {cfg.output_dir} after period {pid} download")

for raw_out in raw_files:
    data_date = _extract_data_date(raw_out)
    std_out = cfg.output_dir / f"oscar_uv_clip_{data_date}.nc"
    if std_out.exists():
        print(f"[DEBUG] clip exists, skipping: {std_out.name}")
    else:
        print(f"[DEBUG] standardize raw={raw_out.name} -> clip={std_out.name}")
        standardize_oscar_uv_netcdf(raw_out, std_out, area, u_var=cfg.u_var, v_var=cfg.v_var)
    export_oscar_uv_geotiff(std_out, cfg.output_dir, area, u_var=cfg.u_var, v_var=cfg.v_var)

[DEBUG] clip exists, skipping: oscar_uv_clip_20200131.nc
[DEBUG] wrote GeoTIFFs: oscar_uv_clip_20200131_u.tif, oscar_uv_clip_20200131_v.tif
[DEBUG] clip exists, skipping: oscar_uv_clip_20200130.nc
[DEBUG] wrote GeoTIFFs: oscar_uv_clip_20200130_u.tif, oscar_uv_clip_20200130_v.tif
[DEBUG] clip exists, skipping: oscar_uv_clip_20200129.nc
[DEBUG] wrote GeoTIFFs: oscar_uv_clip_20200129_u.tif, oscar_uv_clip_20200129_v.tif
[DEBUG] clip exists, skipping: oscar_uv_clip_20200128.nc
[DEBUG] wrote GeoTIFFs: oscar_uv_clip_20200128_u.tif, oscar_uv_clip_20200128_v.tif
[DEBUG] clip exists, skipping: oscar_uv_clip_20200127.nc
[DEBUG] wrote GeoTIFFs: oscar_uv_clip_20200127_u.tif, oscar_uv_clip_20200127_v.tif
[DEBUG] clip exists, skipping: oscar_uv_clip_20200126.nc
[DEBUG] wrote GeoTIFFs: oscar_uv_clip_20200126_u.tif, oscar_uv_clip_20200126_v.tif
[DEBUG] clip exists, skipping: oscar_uv_clip_20200125.nc
[DEBUG] wrote GeoTIFFs: oscar_uv_clip_20200125_u.tif, oscar_uv_clip_20200125_v.tif
[DEBUG] clip exists,